# 13E - JBHI five-pipeline inventory and coverage

Select Qwen, MedGemma, GPT-OSS and ECG-Instruct from `generation_v3`, and PULSE from
`pulse_compact_v1`, using `analysis_model_runs.json`. No recursive scanning or legacy/rescue pooling.
ECG is primary (four pipelines); CXR is secondary (three). GPT-OSS is text-only.
This audit validates protocol/context/manifest hashes, deduplicates cells, and distinguishes
successful, failed and unattempted cases. Schema echoes marked OK are rejected by the audit.
Generation files are read-only. Stop generation before the final analysis freeze.
The source outline is `revised_manuscript/jbhi_outline.txt`; these are exploratory analyses,
not retrospectively prespecified outcomes.

In [ ]:
from pathlib import Path
import os, sys
CODE_ROOT = Path(os.environ.get('EXPERIMENT4_CODE_ROOT', str(Path.cwd())))
if not (CODE_ROOT / 'e4_common.py').is_file() and 'EXPERIMENT4_CODE_ROOT' not in os.environ:
    CODE_ROOT = Path.cwd()
if not CODE_ROOT.is_absolute() or not (CODE_ROOT / 'e4_common.py').is_file():
    raise FileNotFoundError('Set EXPERIMENT4_CODE_ROOT to the uploaded Experiment 4 source directory')
if str(CODE_ROOT) not in sys.path: sys.path.insert(0, str(CODE_ROOT))
from e4_common import ARTIFACT_ROOT, EXPERIMENT4_ROOT, SYMILE_ROOT
print({'code_root': str(CODE_ROOT), 'artifact_root': str(ARTIFACT_ROOT),
       'experiment4_root': str(EXPERIMENT4_ROOT), 'symile_root': str(SYMILE_ROOT)})

In [ ]:
# Same selection file and analysis ID in notebooks 13E-16; independent of generation run env.
os.environ['E4_ANALYSIS_CONFIG'] = str(CODE_ROOT / 'analysis_model_runs.json')
os.environ.setdefault('E4_ANALYSIS_ID', 'jbhi_five_models_v1')
from e4_jbhi_analysis import configuration, analysis_path
config = configuration()
print('Model-to-run mapping:', config['model_runs'])
print('Analysis output:', analysis_path(EXPERIMENT4_ROOT, config))

In [ ]:
from e4_jbhi_analysis import audit_generation
coverage = audit_generation()
display(coverage)
display(coverage.groupby(['model_key', 'direction'])[['expected', 'successful', 'failed', 'pending']].sum())
print('Failed cells remain in availability denominators. Pending cells indicate unfinished generation.')